In [ ]:
from go_bpe import BPETokenizer

In [ ]:
class GPTConfig:
    def __init__(
        self,
        transformer_blocks,
        emb_size,
        vocab_size,
        context_size,
        d_in,
        d_out,
        heads_len,
        dropout,
    ):
        self.transformer_blocks = transformer_blocks
        self.vocab_size = vocab_size
        self.emb_size = emb_size
        self.context_size = context_size
        self.d_in = d_in
        self.d_out = d_out
        self.heads_len = heads_len
        self.dropout = dropout

In [ ]:
vocab_size = 1024
context_size = 64
emb_size = 768

cfg = GPTConfig(
    transformer_blocks=4,
    emb_size=emb_size,
    vocab_size=vocab_size,
    context_size=context_size,
    heads_len=12,
    d_in=emb_size,
    d_out=emb_size,
    dropout=0.1,
)


In [ ]:
with open("./the-verdict.txt", "r", encoding="utf-8") as f:
    verdict_corpus = f.read()

In [ ]:
tok = BPETokenizer.train(verdict_corpus, max_vocab_size=vocab_size)
tok.save("model.json")

In [ ]:
with BPETokenizer.load("model.json") as tok:
    ids = tok.encode(verdict_corpus)
    text = tok.decode(ids)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class JohnnyDataset(Dataset):
    def __init__(self, txt, tok, context_size, shift):
        self.input_ids = []
        self.target_ids = []
        token_ids = tok.encode(txt)

        for i in range(0, len(token_ids) - context_size, shift):
            input_id = token_ids[i : i + context_size]
            target_id = token_ids[i + 1 : i + context_size + 1]
            self.input_ids.append(torch.tensor(input_id))
            self.target_ids.append(torch.tensor(target_id))

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

    def __len__(self):
        return len(self.target_ids)


In [ ]:
def create_dataloader(
    txt,
    vocab_json="model.json",
    context_size=256,
    shift=128,
    batch_size=4,
    drop_last=True,
    num_workers=0,
    shuffle=True,
):
    tok = BPETokenizer.load(vocab_json)
    dataset = JohnnyDataset(txt, tok, context_size, shift)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        drop_last=drop_last,
        num_workers=num_workers,
        shuffle=shuffle,
    )

    return dataloader

In [ ]:
dataloader = create_dataloader(verdict_corpus, context_size=context_size, shift=128, batch_size=8)
data_iter = iter(dataloader)

In [ ]:
inputs, outputs = next(data_iter)
inputs.shape

In [ ]:
class MultiHeadSelfAttention(torch.nn.Module):
    mask: torch.Tensor

    def __init__(self, cfg: GPTConfig, bias=False):
        super().__init__()
        self.head_out = cfg.d_out // cfg.heads_len  # 8 / 2 = 4
        self.heads_len = cfg.heads_len
        self.d_out = cfg.d_out
        self.qW = torch.nn.Linear(cfg.d_in, cfg.d_out, bias=bias)  # 3, 8
        self.kW = torch.nn.Linear(cfg.d_in, cfg.d_out, bias=bias)
        self.vW = torch.nn.Linear(cfg.d_in, cfg.d_out, bias=bias)
        self.dropout = torch.nn.Dropout(cfg.dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_size, context_size), diagonal=1)
        )

    def forward(self, x):
        b, context_size, d_in = x.shape  # 2, 4, 3
        queries = self.qW(x)
        keys = self.kW(x)
        values = self.vW(x)  # 2, 4, 8

        queries = queries.view(b, context_size, self.heads_len, self.head_out)
        keys = keys.view(b, context_size, self.heads_len, self.head_out)
        values = values.view(b, context_size, self.heads_len, self.head_out)

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)  # 2,2,3,4

        attention_scores = queries @ keys.transpose(2, 3)  # 2,2,3,4 @ 2,2,4,3 = 2,2,3,3
        mask = self.mask.bool()[:context_size, :context_size]
        attention_scores.masked_fill_(mask, -torch.inf)
        attention_w = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim=-1)

        attention_w = self.dropout(attention_w)
        context = attention_w @ values  # 2,2,3,4

        context = context.transpose(1, 2)
        context = context.contiguous().view(b, context_size, self.d_out)

        return context


In [ ]:
class LayerNorm(torch.nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.eps = 1e-5
        self.scale = torch.nn.Parameter(torch.ones(cfg.emb_size))
        self.shift = torch.nn.Parameter(torch.zeros(cfg.emb_size))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * x + self.shift

In [ ]:
class GELU(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return (
            0.5
            * x
            * (
                1
                + torch.tanh(
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )

In [ ]:
class FFNetwork(torch.nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(cfg.emb_size, 4 * cfg.emb_size),
            GELU(),
            torch.nn.Linear(4 * cfg.emb_size, cfg.emb_size),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class Transformer(torch.nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.mhsa = MultiHeadSelfAttention(cfg)
        self.preNorm = LayerNorm(cfg)
        self.postNorm = LayerNorm(cfg)
        self.FF = FFNetwork(cfg)
        self.drop = torch.nn.Dropout(cfg.dropout)

    def forward(self, x):
        skip = x
        x = self.preNorm(x)
        x = self.mhsa(x)
        x = self.drop(x)
        x = x + skip

        skip = x
        x = self.postNorm(x)
        x = self.FF(x)
        x = self.drop(x)
        x = x + skip

        return x

In [ ]:
class JohnnyGPT(torch.nn.Module):
    def __init__(self, cfg:GPTConfig):
        super().__init__()
        self.token_emb = torch.nn.Embedding(cfg.vocab_size, cfg.emb_size)
        self.pos_emb = torch.nn.Embedding(cfg.context_size, cfg.emb_size)
        self.drop = torch.nn.Dropout(cfg.dropout)
        self.trans_blocks = torch.nn.Sequential(*[Transformer(cfg) for _ in range(cfg.transformer_blocks)])
        self.norm = LayerNorm(cfg)
        self.out = torch.nn.Linear(cfg.emb_size, cfg.vocab_size)

    def forward(self, x):
        batch_size, context_size = x.shape
        token_emb = self.token_emb(x)
        pos_emb = self.pos_emb(torch.arange(context_size, device=x.device))

        x = token_emb + pos_emb
        x = self.drop(x)
        x = self.trans_blocks(x)
        x = self.norm(x)
        x = self.out(x)

        return x


In [ ]:
model = JohnnyGPT(cfg)

In [ ]:
model.eval()
def simple_generator(model, idx, context_size, max_tokens):
    for _ in range(max_tokens):
        idx_trim = idx[:, -context_size:]
        with torch.no_grad():
            out = model(idx_trim)

        out = out[:, -1, :]
        prob = torch.softmax(out, dim=-1)
        idx_next = torch.argmax(prob, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=-1)

    return idx




In [ ]:
tok = BPETokenizer.load("model.json")

test = "Hello, how are"
encodded = tok.encode(test)
tensor = torch.tensor(encodded, dtype=torch.long).unsqueeze(0)

out = simple_generator(model, tensor, cfg.context_size, 20)
print(out)
print(tok.decode(out.squeeze().tolist()))

In [ ]:
total_params = sum([p.numel() for p in model.parameters()])
print(f"Total number of parameters: {total_params:,}")
print(f"Total model weight: {(total_params*4)//1024//1024} Mb")

In [ ]:
def train(model, train_loader, test_loader, optimizer, epochs):
    for _ in range(epochs):
        model.train()

        for x_train, y_train in train_loader:
            optimizer.zero_grad()
            out = model(x_train)
            loss = torch.nn.functional.cross_entropy(out.flatten(0, 1), y_train.flatten())
            loss.backward()
            optimizer.step()

            print(loss)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
epochs = 100
train(model, dataloader, dataloader, optimizer, epochs)